In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System

/content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System


In [3]:
!pwd

/content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System


In [4]:
!ls

data  notebooks  saved_models  src  tmp_trainer


In [5]:
!pip install -q sentence-transformers

##Create MPNet Folders

In [6]:
!mkdir -p src/models/mpnet
!mkdir -p saved_models/mpnet

In [7]:
!ls src/models

bert  distilbert  mpnet  roberta  sentence_bert


In [8]:
!ls saved_models

bert  distilbert  mpnet  roberta  sentence_bert


In [9]:
%%writefile src/models/mpnet/train.py

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-mpnet-base-v2"


def load_model():

    print("Loading MPNet model...")

    model = SentenceTransformer(MODEL_NAME)

    print("MPNet model loaded successfully")

    return model


if __name__ == "__main__":

    load_model()

Writing src/models/mpnet/train.py


In [6]:
!python src/models/mpnet/train.py

Loading MPNet model...
modules.json: 100% 349/349 [00:00<00:00, 1.67MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 531kB/s]
README.md: 100% 11.6k/11.6k [00:00<00:00, 31.8MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 366kB/s]
config.json: 100% 571/571 [00:00<00:00, 3.31MB/s]
model.safetensors: 100% 438M/438M [00:03<00:00, 124MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 7695.83it/s]
tokenizer_config.json: 100% 363/363 [00:00<00:00, 2.45MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 67.7MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 91.3MB/s]
special_tokens_map.json: 100% 239/239 [00:00<00:00, 1.41MB/s]
config.json: 100% 190/190 [00:00<00:00, 962kB/s]
MPNet model loaded successfully


In [10]:
%%writefile src/models/mpnet/generate_embeddings.py

import os
import torch
import pandas as pd

from sentence_transformers import SentenceTransformer


MODEL_NAME = "all-mpnet-base-v2"


def generate_embeddings():

    print("Loading job dataset...")

    df = pd.read_csv(
        "data/processed(job matcher)/processed_jobs.csv"
    )

    print(f"Dataset Shape: {df.shape}")


    print("Loading MPNet model...")

    model = SentenceTransformer(MODEL_NAME)


    print("Generating job embeddings...")


    embeddings = model.encode(
        df["clean_job_text"].tolist(),
        batch_size=16,
        show_progress_bar=True,
        convert_to_tensor=True
    )


    os.makedirs(
        "saved_models/mpnet",
        exist_ok=True
    )


    torch.save(
        embeddings,
        "saved_models/mpnet/job_embeddings.pt"
    )


    df.to_csv(
        "saved_models/mpnet/job_metadata.csv",
        index=False
    )


    print("\nEmbeddings generated successfully!")
    print("Saved:")
    print(" - job_embeddings.pt")
    print(" - job_metadata.csv")


if __name__ == "__main__":

    generate_embeddings()

Writing src/models/mpnet/generate_embeddings.py


In [7]:
!python src/models/mpnet/generate_embeddings.py

Loading job dataset...
Dataset Shape: (110898, 7)
Loading MPNet model...
Loading weights: 100% 199/199 [00:00<00:00, 739.11it/s]
Generating job embeddings...
Batches: 100% 6932/6932 [40:37<00:00,  2.84it/s]

Embeddings generated successfully!
Saved:
 - job_embeddings.pt
 - job_metadata.csv


In [11]:
%%writefile src/models/mpnet/matcher.py

import torch
import pandas as pd

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim


MODEL_NAME = "all-mpnet-base-v2"


class JobMatcher:


    def __init__(self):

        print("Loading MPNet model...")

        self.model = SentenceTransformer(MODEL_NAME)


        self.job_embeddings = torch.load(
            "saved_models/mpnet/job_embeddings.pt"
        )


        self.jobs = pd.read_csv(
            "saved_models/mpnet/job_metadata.csv"
        )


        print("Matcher ready.")



    def recommend(self, resume_text, top_k=5):


        resume_embedding = self.model.encode(
            resume_text,
            convert_to_tensor=True
        )


        similarity_scores = cos_sim(
            resume_embedding,
            self.job_embeddings
        )[0]


        top_results = torch.topk(
            similarity_scores,
            k=top_k
        )


        recommendations = []


        for score, index in zip(
            top_results.values,
            top_results.indices
        ):


            job = self.jobs.iloc[index.item()]


            recommendations.append({

                "job_title": job["title"],

                "industry": job["industry_name"],

                "match_score": round(
                    score.item() * 100,
                    2
                )

            })


        return recommendations



if __name__ == "__main__":


    matcher = JobMatcher()


    resume = """

    Python developer with experience in
    Machine Learning,
    Deep Learning,
    NLP,
    SQL,
    TensorFlow,
    Data Analysis

    """


    jobs = matcher.recommend(
        resume,
        top_k=5
    )


    print("\nTop Recommended Jobs\n")


    for i, job in enumerate(jobs, 1):

        print(f"{i}. {job}")

Writing src/models/mpnet/matcher.py


In [8]:
!python src/models/mpnet/matcher.py

Loading MPNet model...
Loading weights: 100% 199/199 [00:00<00:00, 5372.19it/s]
Matcher ready.

Top Recommended Jobs

1. {'job_title': 'Python Developer with AI/ML', 'industry': 'IT Services and IT Consulting', 'match_score': 79.17}
2. {'job_title': 'Python Developer with AI/ML Skills', 'industry': 'IT Services and IT Consulting', 'match_score': 79.11}
3. {'job_title': 'Sr. Python Developer with Devops exp- W2 only', 'industry': 'Hospitality, IT Services and IT Consulting', 'match_score': 75.76}
4. {'job_title': 'Python Developer', 'industry': 'IT Services and IT Consulting', 'match_score': 75.49}
5. {'job_title': 'Python Developer', 'industry': 'Technology, Information and Internet', 'match_score': 75.26}


In [12]:
%%writefile src/models/mpnet/evaluate.py

from matcher import JobMatcher


def evaluate():

    print("Initializing MPNet Job Matcher...\n")

    matcher = JobMatcher()


    resume = """

    Experienced Python developer with expertise in
    Machine Learning,
    Deep Learning,
    Natural Language Processing,
    SQL,
    TensorFlow,
    Data Analysis,
    REST APIs,
    Flask,
    Django.

    """


    recommendations = matcher.recommend(
        resume_text=resume,
        top_k=5
    )


    print("\n========== TOP 5 RECOMMENDED JOBS ==========\n")


    for i, job in enumerate(recommendations, start=1):

        print(f"{i}. Job Title   : {job['job_title']}")
        print(f"   Industry   : {job['industry']}")
        print(f"   Match Score: {job['match_score']}%")
        print("-" * 45)


if __name__ == "__main__":

    evaluate()

Overwriting src/models/mpnet/evaluate.py


In [13]:
!python src/models/mpnet/evaluate.py

Initializing MPNet Job Matcher...

Loading MPNet model...
Loading weights: 100% 199/199 [00:00<00:00, 5884.56it/s]
Matcher ready.

========== TOP 5 RECOMMENDED JOBS ==========

1. Job Title   : Python Developer with AI/ML
   Industry   : IT Services and IT Consulting
   Match Score: 79.26%
---------------------------------------------
2. Job Title   : Python Developer
   Industry   : IT Services and IT Consulting
   Match Score: 79.24%
---------------------------------------------
3. Job Title   : Sr. Python Developer
   Industry   : Human Resources Services
   Match Score: 78.81%
---------------------------------------------
4. Job Title   : Python Developer with AI/ML Skills
   Industry   : IT Services and IT Consulting
   Match Score: 78.79%
---------------------------------------------
5. Job Title   : Python Developer
   Industry   : IT Services and IT Consulting
   Match Score: 78.59%
---------------------------------------------
